# Create CSV Files for Kenny's Analysis

This notebook creates two CSV files:
1. **LRI Factor Loadings**: Rows = LRI tuples (sender, receiver, ligand, receptor), Columns = motifs
2. **Differential Expression Results**: Rows = (cell type, gene) pairs, Columns = motifs (coefficients + p-values)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

## Configuration

In [ ]:
# Update these paths to match your data
BPTF_DIR = '../results/bptf'  # Directory with BPTF results
GLM_DIR = '../results/glm'    # Directory with GLM results  
OUTPUT_DIR = '../results/kenny_csvs'  # Output directory
SPLITTER = '|'  # Separator used in LRI names

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Create LRI Factor Loadings CSV

In [ ]:
# Load BPTF results
print("Loading BPTF results...")
lri_factors = np.load(os.path.join(BPTF_DIR, 'lri_factors.npy'))  # motifs x LRIs
lri_motifs_df = pd.read_csv(os.path.join(BPTF_DIR, 'lri_motifs.csv'))

print(f"LRI factors shape: {lri_factors.shape}")
print(f"LRI motifs data: {len(lri_motifs_df)} entries")

In [ ]:
# Parse LRI names and create factor loadings CSV
print("Creating LRI factor loadings CSV...")

# Get unique LRI names
unique_lris = lri_motifs_df['lri_name'].unique()
n_motifs = lri_factors.shape[0]

# Parse LRI names into components
lri_data = []
for lri_name in unique_lris:
    parts = lri_name.split(SPLITTER)
    if len(parts) >= 4:
        sender = parts[0]
        receiver = parts[1] 
        ligand = parts[2]
        receptor = parts[3]
        
        # Get factor loadings for this LRI across all motifs
        lri_idx = lri_motifs_df[lri_motifs_df['lri_name'] == lri_name]['lri_idx'].iloc[0]
        
        # Create row with LRI info + factor loadings
        row = {
            'Sender': sender,
            'Receiver': receiver,
            'Ligand': ligand,
            'Receptor': receptor
        }
        
        # Add factor loadings for each motif
        for motif_idx in range(n_motifs):
            row[f'Motif_{motif_idx}'] = lri_factors[motif_idx, lri_idx]
            
        lri_data.append(row)

# Create DataFrame and save
lri_df = pd.DataFrame(lri_data)
lri_csv_path = os.path.join(OUTPUT_DIR, "lri_factor_loadings.csv")
lri_df.to_csv(lri_csv_path, index=False)

print(f"LRI factor loadings CSV saved: {lri_csv_path}")
print(f"Shape: {lri_df.shape}")
lri_df.head()

## 2. Create Differential Expression Results CSV

In [ ]:
# Load GLM results
print("Loading GLM results...")
glm_files = glob.glob(os.path.join(GLM_DIR, "motif_*_celltype_*_de_results.csv"))

if not glm_files:
    print(f"No GLM results found in {GLM_DIR}")
else:
    print(f"Found {len(glm_files)} GLM result files")

# Parse file names to get motif and cell type info
glm_data = {}

for file_path in glm_files:
    filename = os.path.basename(file_path)
    # Parse: motif_{X}_celltype_{Y}_de_results.csv
    parts = filename.replace('.csv', '').split('_')
    
    try:
        motif_idx = int(parts[1])  # motif_X -> X
        celltype_start = parts.index('celltype') + 1
        celltype_parts = parts[celltype_start:-2]  # everything between celltype and 'de results'
        celltype = '_'.join(celltype_parts)
        
        # Load the results
        df = pd.read_csv(file_path)
        
        if motif_idx not in glm_data:
            glm_data[motif_idx] = {}
        glm_data[motif_idx][celltype] = df
        
    except (ValueError, IndexError) as e:
        print(f"Warning: Could not parse filename {filename}: {e}")
        continue

print(f"Loaded GLM data for {len(glm_data)} motifs")

In [ ]:
# Create DE results CSV
if glm_data:
    print("Creating differential expression results CSV...")
    
    # Collect all unique (cell type, gene) pairs
    all_pairs = set()
    motif_indices = sorted(glm_data.keys())
    
    for motif_idx in motif_indices:
        for celltype, df in glm_data[motif_idx].items():
            for gene in df['gene']:
                all_pairs.add((celltype, gene))
    
    print(f"Found {len(all_pairs)} unique (cell type, gene) pairs")
    
    # Create base DataFrame
    pairs_list = list(all_pairs)
    de_df = pd.DataFrame(pairs_list, columns=['Cell_Type', 'Gene'])
    
    # Add columns for each motif
    for motif_idx in motif_indices:
        coeff_col = f'Motif_{motif_idx}_Coefficient'
        pval_col = f'Motif_{motif_idx}_PValue'
        
        de_df[coeff_col] = np.nan
        de_df[pval_col] = np.nan
        
        # Fill in data for this motif
        for celltype, df in glm_data[motif_idx].items():
            for _, row in df.iterrows():
                gene = row['gene']
                
                # Find matching row in de_df
                mask = (de_df['Cell_Type'] == celltype) & (de_df['Gene'] == gene)
                
                if mask.any():
                    de_df.loc[mask, coeff_col] = row.get('logFC', np.nan)
                    de_df.loc[mask, pval_col] = row.get('pval', row.get('qval', np.nan))
    
    # Save CSV
    de_csv_path = os.path.join(OUTPUT_DIR, "differential_expression_results.csv")
    de_df.to_csv(de_csv_path, index=False)
    
    print(f"DE results CSV saved: {de_csv_path}")
    print(f"Shape: {de_df.shape}")
    
else:
    print("No GLM data available - skipping DE results CSV")

## Summary

In [ ]:
print("="*60)
print("CSV CREATION COMPLETED!")
print("="*60)
print(f"Files created in: {OUTPUT_DIR}")
print("\nFiles:")
print(f"  1. lri_factor_loadings.csv")
if glm_data:
    print(f"  2. differential_expression_results.csv")

print(f"\nSummary:")
print(f"  - LRI tuples: {len(lri_df)}")
print(f"  - Motifs: {lri_factors.shape[0]}")
if glm_data:
    print(f"  - (Cell type, gene) pairs: {len(de_df)}")
    print(f"  - Motifs with DE data: {len(glm_data)}")

In [ ]:
# Preview the files
print("LRI Factor Loadings (first 5 rows):")
print(lri_df.head())

if 'de_df' in globals():
    print("\nDifferential Expression Results (first 5 rows):")
    print(de_df.head())